<a href="https://colab.research.google.com/github/Odewenu/network-anomaly-detection/blob/main/notebooks/01_data_cleaning_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Phase 1: UNSW-NB15 data cleaning
!pip install -q kagglehub pyarrow

import os
import numpy as np
import pandas as pd
import joblib
import kagglehub
from sklearn.preprocessing import StandardScaler

# 1. Load the data
folder = kagglehub.dataset_download("dhoogla/unswnb15")
train = pd.read_parquet(os.path.join(folder, "UNSW_NB15_training-set.parquet"))
test = pd.read_parquet(os.path.join(folder, "UNSW_NB15_testing-set.parquet"))
print("Loaded:", train.shape, test.shape)
print("Missing values:", train.isnull().sum().sum(), test.isnull().sum().sum())

# Convert text-type columns to plain text
cat_cols = ["proto", "service", "state"]
for df in (train, test):
    for c in cat_cols + ["attack_cat"]:
        df[c] = df[c].astype(str)

# 2. Remove duplicate rows
print("Duplicates:", train.duplicated().sum(), test.duplicated().sum())
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", train.shape, test.shape)
print(train["label"].value_counts())

# 3. Log-transform heavily skewed columns
num_cols = train.select_dtypes(include="number").columns.drop("label")
skew = train[num_cols].skew()
skewed_cols = [c for c in skew[skew > 5].index
               if train[c].min() >= 0 and test[c].min() >= 0
               and train[c].nunique() > 2]
for df in (train, test):
    df[skewed_cols] = np.log1p(df[skewed_cols])
print("Log-transformed", len(skewed_cols), "columns")

# 4. Group rare protocols, then one-hot encode text columns
top_proto = train["proto"].value_counts().head(5).index
for df in (train, test):
    df["proto"] = df["proto"].where(df["proto"].isin(top_proto), "other")
train = pd.get_dummies(train, columns=cat_cols, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, dtype=int)
test = test.reindex(columns=train.columns, fill_value=0)
print("After encoding:", train.shape, test.shape)

# 5. Scale numeric columns (fit on train only)
to_scale = [c for c in train.columns
            if c not in ("label", "attack_cat") and train[c].nunique() > 2]
scaler = StandardScaler()
train[to_scale] = scaler.fit_transform(train[to_scale])
test[to_scale] = scaler.transform(test[to_scale])
print("Scaled", len(to_scale), "columns")

# 6. Save the cleaned data
os.makedirs("data/processed", exist_ok=True)

def small(df):
    df = df.copy()
    f = df.select_dtypes("float64").columns
    df[f] = df[f].astype("float32")
    return df

small(train).to_parquet("data/processed/train_clean.parquet", index=False, compression="gzip")
small(test).to_parquet("data/processed/test_clean.parquet", index=False, compression="gzip")
joblib.dump(scaler, "data/processed/scaler.pkl")
pd.DataFrame({"column": train.columns, "type": train.dtypes.astype(str).values}) \
    .to_csv("data/processed/data_schema.csv", index=False)

for f in os.listdir("data/processed"):
    print(f, round(os.path.getsize("data/processed/" + f) / 1e6, 1), "MB")

100%|██████████| 11.7M/11.7M [00:00<00:00, 104MB/s] 

Extracting files...


Loaded: (175341, 36) (82332, 36)
Missing values: 0 0
Duplicates: 78519 32361
After removing duplicates: (96822, 36) (49971, 36)
label
0    48894
1    47928
Name: count, dtype: int64
Log-transformed 21 columns
After encoding: (96822, 61) (49971, 61)
Scaled 30 columns
train_clean.parquet 6.3 MB
data_schema.csv 0.0 MB
scaler.pkl 0.0 MB
test_clean.parquet 3.3 MB
